# HUM-450 Data preprocessing
The goal of this notebooks is to clean and concatenate the data in order to generate the database

In [ ]:
import numpy as np
import pandas as pd
import os
import re
from collections import Counter, defaultdict
import networkx as nx
import folium
from folium.plugins import MarkerCluster
import geopy


#Path to the data folder. Here is what you need to modify
ROOT="/Users/edouardkoehn/Documents/Github/HUM-450"
time_windows=["1890-1910", "1945-1965"]

In [ ]:
def load_data(csv_path: str, date: str, query_words: str, queries_used) -> pd.DataFrame:
    """
    Return the data contained in the CSV file as a DataFrame. It selects specific columns and filters the data.

    Parameters:
    - csv_path (str): The file path of the CSV file.
    - date (str): The date string to assign to the 'time_window' column.
    - query_word (list): The query words to assign to the 'query_word' column.
    - queries_used(list):all the query used for defining the dataset

    Returns:
    - data (pd.DataFrame): The DataFrame containing the loaded and filtered data.
    """
    data = pd.read_csv(csv_path,
                       sep=';',
                       dtype=pd.StringDtype(),
                       usecols=['uid', 'language', 'title', 'size', 'country', 'newspaper', 'pages', 'nb_pages',
                                'year', 'is_on_front', 'date', 'persons_mentioned', 'locations_mentioned', 
                                'content', 'access_right']
                      )
    for query in queries_used:
        if(query in query_words):
            data[f'{query}'] = np.ones(data.shape[0])
        else:
            data[f'{query}'] = np.zeros(data.shape[0])
    data['time_window'] = date
    data = data.fillna(pd.NA)

    data = filter_location(data)
    data = filter_people(data)
    data = filter_content(data)
    data=filter_title(data)
    data=data.dropna(ignore_index=True)

    return data

def filter_location(data: pd.DataFrame) -> pd.DataFrame:
    """
    Filter the 'locations_mentioned' column of the DataFrame by splitting the locations string into a list of locations.

    Parameters:
    - data (pd.DataFrame): The DataFrame containing the 'locations_mentioned' column.

    Returns:
    - data (pd.DataFrame): The DataFrame with the 'locations_mentioned' column processed.
    """
    data['locations_mentioned'] = [location.split('|') if pd.isna(location) == False else np.nan for location in data['locations_mentioned'].to_numpy()]
    data['locations_mentioned']=np.array(data['locations_mentioned'])
    return data

def filter_people(data: pd.DataFrame) -> pd.DataFrame:
    """
    Filter the 'persons_mentioned' column of the DataFrame by splitting the persons string into a list of persons.

    Parameters:
    - data (pd.DataFrame): The DataFrame containing the 'persons_mentioned' column.

    Returns:
    - data (pd.DataFrame): The DataFrame with the 'persons_mentioned' column processed.
    """
    data['persons_mentioned'] = [location.split('|') if pd.isna(location) == False else np.nan for location in data['persons_mentioned'].to_numpy()]
    data['persons_mentioned']=np.array(data['persons_mentioned'])
    return data

def filter_content(data: pd.DataFrame) -> pd.DataFrame:
    """
    Filter the 'content' column of the DataFrame by removing special characters and converting to lowercase.

    Parameters:
    - data (pd.DataFrame): The DataFrame containing the 'content' column.

    Returns:
    - data (pd.DataFrame): The DataFrame with the 'content' column processed.
    """
    spec_char = r'[();,"|%\'\'$*&:«».#^/<>+-]'
    data['content'] = [content.lower() for content in data['content'].to_numpy()]
    data['content'] = [re.sub(spec_char, ' ', content) for content in data['content'].to_numpy()]
    return data

def filter_title(data: pd.DataFrame) -> pd.DataFrame:
    """
    Filter the 'content' column of the DataFrame by removing special characters and converting to lowercase.

    Parameters:
    - data (pd.DataFrame): The DataFrame containing the 'content' column.

    Returns:
    - data (pd.DataFrame): The DataFrame with the 'content' column processed.
    """
    spec_char = r'[();,"|%\'\'$*&:«».#^/<>+-]'
    data['title'] = [content.lower() if pd.isna(content) == False else np.nan for content in data['title'].to_numpy()]
    data['title'] = [re.sub(spec_char, ' ', content)  if pd.isna(content) == False else np.nan for content in data['title'].to_numpy()]
    return data

def export_database(ouptut_path:str, data:pd.DataFrame , queries_used:list):
    """
    Generate the csv reprsenting the data set.

    Parameters:
    - output_path(str): path of the output path
    - data(pd.Dataframe): the df containing the dataset
    - queries_used(list): query word used for generating the dataset
    Returns:
    """
    data.to_csv(os.path.join(ouptut_path,"database.csv"), sep=';', index=False)
    pd.DataFrame(queries_used, columns=['Query']).to_csv(os.path.join(ouptut_path,"query_words.csv"), sep=';', index=False)
    return

In [ ]:
#Extract the file path
path_data=ROOT+"/data"

extraction_path={}
for window in time_windows:
    extraction_path[f"{window}"]=[os.path.join(path_data,window,file) for file in os.listdir(path_data+f"/{window}")]


#Find all the queries used in the datset
queries_used=[]
for window in time_windows:
    for file in extraction_path[f"{window}"]:
        queries=os.path.basename(file)[:-4].split('_')[2:]
        queries_used+=queries

In [ ]:
#Import and clean the data
data=[]
for window in time_windows:
    for file in extraction_path[f"{window}"]:
        queries=os.path.basename(file)[:-4].split('_')[2:]
        data.append(load_data(file, window,queries, queries_used))
extraction=pd.concat(data)
print('dataset_size:',extraction.shape)
extraction.head()

In [ ]:
export_database(path_data, extraction, queries_used)
